# Circumplex Model - Apply Russell's Model to Reddit Posts

Apply Russell's Circumplex Model (Valence × Arousal) to 40K Reddit posts.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

from emotion_circumplex_mapping import calculate_vad, get_quadrant, get_intensity

# Detect environment and set paths
if os.path.exists('/content/drive'):
    BASE_DIR = Path("/content/drive/MyDrive/mental_health_research_V2")
else:
    # Local fallback
    BASE_DIR = Path.cwd()

RESULTS_DIR = BASE_DIR / "results" / "reddit_inference"

print(f"Base directory: {BASE_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Directory exists: {RESULTS_DIR.exists()}")

Mounted at /content/drive
Base directory: /content/drive/MyDrive/mental_health_research_V2
Results directory: /content/drive/MyDrive/mental_health_research_V2/results/reddit_inference
Directory exists: True


In [2]:
# Load emotion predictions
files = sorted(RESULTS_DIR.glob("reddit_emotions_predicted_*.json"))

if not files:
    print(f"ERROR: No emotion prediction files in {RESULTS_DIR}")
    raise FileNotFoundError("Run Mental_Health_Reddit_Inference.ipynb first")

input_file = files[-1]
print(f"Loading: {input_file.name}")

df = pd.read_json(input_file)
print(f"Loaded {len(df):,} posts")

Loading: reddit_emotions_predicted_20251118_194626.json
Loaded 40,745 posts


In [3]:
# Calculate circumplex scores using dominant emotion method
valence_list = []
arousal_list = []
dominance_list = []
quadrant_list = []
intensity_list = []

for emotion_probs in tqdm(df['emotion_probabilities'], desc="Calculating VAD"):
    # Use dominant emotion method (not weighted) to avoid arousal bias
    scores = calculate_vad(emotion_probs, use_dominance=True, method='dominant')

    v = scores['valence']
    a = scores['arousal']
    d = scores['dominance']

    valence_list.append(v)
    arousal_list.append(a)
    dominance_list.append(d)
    quadrant_list.append(get_quadrant(v, a))
    intensity_list.append(get_intensity(v, a))

# Add to dataframe
df['valence'] = valence_list
df['arousal'] = arousal_list
df['dominance'] = dominance_list
df['quadrant'] = quadrant_list
df['intensity'] = intensity_list

print(f"Done. Shape: {df.shape}")

Calculating VAD:   0%|          | 0/40745 [00:00<?, ?it/s]

Done. Shape: (40745, 35)


In [4]:
# Summary statistics
print("="*50)
print("CIRCUMPLEX SCORES")
print("="*50)

print("\nValence:")
print(df['valence'].describe())

print("\nArousal:")
print(df['arousal'].describe())

print("\nQuadrant distribution:")
for quad, count in df['quadrant'].value_counts().sort_index().items():
    pct = (count / len(df)) * 100
    print(f"  {quad}: {count:,} ({pct:.1f}%)")

CIRCUMPLEX SCORES

Valence:
count    40745.000000
mean        -0.308550
std          0.491883
min         -0.900000
25%         -0.650000
50%         -0.500000
75%         -0.250000
max          0.900000
Name: valence, dtype: float64

Arousal:
count    40745.000000
mean         0.396753
std          0.328495
min         -0.300000
25%          0.350000
50%          0.550000
75%          0.600000
max          0.800000
Name: arousal, dtype: float64

Quadrant distribution:
  Q1-Excited: 9,282 (22.8%)
  Q2-Distressed: 24,850 (61.0%)
  Q3-Depressed: 5,888 (14.5%)
  Q4-Relaxed: 725 (1.8%)


In [5]:
# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = RESULTS_DIR / f"reddit_with_circumplex_{timestamp}.json"

print(f"Saving to: {output_file}")
df.to_json(output_file, orient='records', indent=2)

file_size_mb = output_file.stat().st_size / (1024 * 1024)
print(f"✓ Saved {len(df):,} posts ({file_size_mb:.1f} MB)")
print(f"\nDataset includes: valence, arousal, dominance, quadrant, intensity")

Saving to: /content/drive/MyDrive/mental_health_research_V2/results/reddit_inference/reddit_with_circumplex_20251120_192036.json
✓ Saved 40,745 posts (236.1 MB)

Dataset includes: valence, arousal, dominance, quadrant, intensity
